In [2]:
import pandas as pd

dataset = pd.read_csv("curated_dataset.csv")
dataset

,Unnamed: 0,Sequence,SMILES,Binding,Enzyme ID,Substrate ID,Publication,Validated,RHEA_ID,EC number,reaction_SMILES
0,0,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,C(C(=O)O)N,0.0,P00509,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN
1,1,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,C[C@@H](C(=O)O)N,1.0,P00509,compound_aminotransferase_dataset 2,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN
2,2,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,CC(C)[C@@H](C(=O)O)N,0.0,P00509,compound_aminotransferase_dataset 3,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN
3,3,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,CC(C)C[C@@H](C(=O)O)N,1.0,P00509,compound_aminotransferase_dataset 4,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN
4,4,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,CC[C@H](C)[C@@H](C(=O)O)N,0.0,P00509,compound_aminotransferase_dataset 5,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
73225,76466,MKYRAVTLESFGYQLAPVVVSTSDLEARLEPLYRQLRIAPGQLQAM...,C1=CC=C(C=C1)C(=O)OC2=CC=C(C=C2)[N+](=O)[O-],0.0,Pelobacter_propionicus,benzoate,https://doi.org/10.1093/synbio/ysaa004,True,NaN,NaN,NaN
73226,76467,MAFLSVNNVEIVGLAAAVPKNVETLDNLEFFAPGEAEKVMALTGIK...,C1=CC=C(C=C1)C(=O)OC2=CC=C(C=C2)[N+](=O)[O-],0.0,Bacteroides_luti,benzoate,https://doi.org/10.1093/synbio/ysaa004,True,NaN,NaN,NaN
73227,76468,MSAPRYSQVSAVAVRLPDEDLTTPELEELLAERNPRVDVPRGLIER...,C1=CC=C(C=C1)C(=O)OC2=CC=C(C=C2)[N+](=O)[O-],0.0,Nonomuraea_candida,benzoate,https://doi.org/10.1093/synbio/ysaa004,True,NaN,NaN,NaN
73228,76469,MRYQRVFINKIAYELPKEKVATSFLEEQLTDVYQELGIPLGQVEAL...,C1=CC=C(C=C1)C(=O)OC2=CC=C(C=C2)[N+](=O)[O-],0.0,Legionella_brunensis,benzoate,https://doi.org/10.1093/synbio/ysaa004,True,NaN,NaN,NaN


In [3]:
# get the inchikeys
from rdkit import Chem
from rdkit.Chem import AllChem

df_unique_compounds = dataset.drop_duplicates(subset=['SMILES']).loc[:, ["Substrate ID", 'SMILES']].reset_index(drop=True)	

def smiles_to_inchikey(smiles):
    if pd.isna(smiles):
        return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    inchikey = AllChem.MolToInchiKey(mol)
    return inchikey

df_unique_compounds['inchikey'] = df_unique_compounds['SMILES'].apply(smiles_to_inchikey)
# separate in different cells for clarity by -
df_unique_compounds["inchikey_skeleton"] = df_unique_compounds['inchikey'].apply(lambda x: x.split('-')[0] if pd.notna(x) else None)
df_unique_compounds["inchikey_stereochemistry"] = df_unique_compounds['inchikey'].apply(lambda x: '-'.join(x.split('-')[:2]) if pd.notna(x) else None)

[15:46:41] WARNING: not removing hydrogen atom without neighbors


In [9]:
unique_compounds = df_unique_compounds.drop_duplicates(subset=['inchikey_skeleton'])

In [11]:
dataset[dataset["Substrate ID"].isin(unique_compounds["Substrate ID"])].to_csv("curated_dataset_no_stereochemistry_duplicates.csv", index=False)